# RO-072 Stanley and rear-wheel feedback: every number in the Note

The car, the course and the trackers live in `images/trackkit.py` (the same file as RO-071's): a kinematic bicycle with
wheelbase 2.5 m and steering at most 26.57°, the test course (10 m straight, a 90° left arc of radius 10 m, 10 m
straight), the start 1.0 m left of the line, trackers at 20 Hz and simulation steps of 0.01 s. Stanley measures its
errors at the front axle, pure pursuit and rear-wheel feedback at the rear axle. Nothing is random, so no seed is needed.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import trackkit as tk
deg = np.degrees

def report(name, run, ref='rear'):
    m = tk.metrics(run, ref)
    print(f"{name:34s} [{ref}] within 0.1 m after {m['reach']:6.2f} m | past the line {m['over']:.3f} m | "
          f"arc: largest {m['arc_peak']:+.3f}, middle {m['arc_mid']:+.4f} | last straight {m['exit_peak']:+.3f} | "
          f"steering up to {m['steer_max']:.1f} deg | {run['t'][-1]:.2f} s")
    return m

## 1. The opening problem: pure pursuit's corner errors (Note, Section 1)

In [2]:
run_pp = tk.simulate(tk.law('pp', ld=4.0), v=tk.V_BASE)
run_st = tk.simulate(tk.law('stanley'), v=tk.V_BASE)
run_rw = tk.simulate(tk.law('rear'), v=tk.V_BASE)
report('pure pursuit 4 m', run_pp)
report('Stanley', run_st, 'front'); report('Stanley', run_st, 'rear')
report('rear-wheel feedback', run_rw)

pure pursuit 4 m                   [rear] within 0.1 m after   7.36 m | past the line 0.000 m | arc: largest +0.120, middle +0.0331 | last straight -0.166 | steering up to 17.4 deg | 17.86 s
Stanley                            [front] within 0.1 m after   9.33 m | past the line 0.000 m | arc: largest +0.079, middle +0.0040 | last straight +0.005 | steering up to 18.4 deg | 17.65 s
Stanley                            [rear] within 0.1 m after  27.94 m | past the line 0.000 m | arc: largest +0.318, middle +0.3167 | last straight +0.235 | steering up to 18.4 deg | 17.65 s
rear-wheel feedback                [rear] within 0.1 m after   4.09 m | past the line 0.039 m | arc: largest -0.006, middle +0.0002 | last straight +0.005 | steering up to 26.6 deg | 17.92 s


{'reach': 4.087278066926159,
 'over': 0.03920083333339354,
 'arc_peak': -0.005707200531654925,
 'arc_mid': 0.00018112311397139904,
 'exit_peak': 0.0046796901112955425,
 'steer_max': 26.56505117707799}

## 2. Stanley at the start (Note, Section 2)

In [3]:
for v in (2.0, 5.0):
    o = tk.stanley(*tk.START, v)
    print(f'{v} m/s: front-axle error {o["e"]:.2f} m, heading term {deg(o["head"]):.2f} deg, cross-track term '
          f'{deg(o["cross"]):.2f} deg, aim point (ks + v)/k = {(tk.KS_ST + v)/tk.K_ST:.1f} m ahead')

2.0 m/s: front-axle error 1.00 m, heading term -0.00 deg, cross-track term -18.43 deg, aim point (ks + v)/k = 3.0 m ahead
5.0 m/s: front-axle error 1.00 m, heading term -0.00 deg, cross-track term -9.46 deg, aim point (ks + v)/k = 6.0 m ahead


In [4]:
# Figure 2: the two terms 0.5 s into the 2 m/s run (the 10th tick)
tick = run_st['ticks'][10]
print(f"t = {tick['t']:.2f} s: front axle at ({tick['fx']:.3f}, {tick['fy']:.3f}), lateral error {tick['e']:.3f} m, "
      f"heading error {deg(tick['the']):.2f} deg")
print(f"heading term {deg(tick['head']):+.2f} deg, cross-track term {deg(tick['cross']):+.2f} deg "
      f"(atan(1 x {tick['e']:.3f} / 3)), steering {deg(tick['delta']):+.2f} deg")

t = 0.50 s: front axle at (3.487, 0.711), lateral error 0.711 m, heading error -5.39 deg
heading term +5.39 deg, cross-track term -13.34 deg (atan(1 x 0.711 / 3)), steering -7.94 deg


## 3. Stanley at 2 and 5 m/s: the front-axle error shrinks at the same rate in time (Note, Section 3)

In [5]:
for v in (2.0, 5.0):
    r = tk.simulate(tk.law('stanley'), v=v)
    rate = tk.K_ST * v / (tk.KS_ST + v)
    for target in (0.5, 0.1):
        i = np.argmax(np.abs(r['ef']) < target)
        print(f'{v} m/s: front error below {target} m after {r["t"][i]:.2f} s, front axle {r["sf"][i]:.2f} m along, '
              f'{r["sf"][i] - r["sf"][0]:.2f} m from its start (straight-line prediction {np.log(1/target)/rate:.2f} s)')
    print('   rate k v/(ks + v) =', round(rate, 4), 'per s; time constant', round(1/rate, 3), 's')
    report(f'Stanley {v} m/s', r, 'front')

2.0 m/s: front error below 0.5 m after 1.03 s, front axle 4.53 m along, 2.03 m from its start (straight-line prediction 1.04 s)
2.0 m/s: front error below 0.1 m after 3.44 s, front axle 9.33 m along, 6.83 m from its start (straight-line prediction 3.45 s)
   rate k v/(ks + v) = 0.6667 per s; time constant 1.5 s
Stanley 2.0 m/s                    [front] within 0.1 m after   9.33 m | past the line 0.000 m | arc: largest +0.079, middle +0.0040 | last straight +0.005 | steering up to 18.4 deg | 17.65 s
5.0 m/s: front error below 0.5 m after 0.81 s, front axle 6.53 m along, 4.03 m from its start (straight-line prediction 0.83 s)
5.0 m/s: front error below 0.1 m after 2.55 s, front axle 15.39 m along, 12.89 m from its start (straight-line prediction 2.76 s)
   rate k v/(ks + v) = 0.8333 per s; time constant 1.2 s
Stanley 5.0 m/s                    [front] within 0.1 m after  15.39 m | past the line 0.000 m | arc: largest +0.276, middle +0.0649 | last straight +0.028 | steering up to 14.7 de

## 4. On the arc: the front axle stays on it, the rear axle cuts inside (Note, Section 3.3)

In [6]:
R_rear = np.sqrt(10**2 - 2.5**2)
print('rear-axle circle when the front axle is on the 10 m arc:', round(R_rear, 3), 'm, so', round(10 - R_rear, 3), 'm inside')
print('steering on the arc: asin(2.5/10) =', round(deg(np.arcsin(0.25)), 2), 'deg; tan form atan(2.5/9.682) =',
      round(deg(np.arctan(2.5 / R_rear)), 2), 'deg')
i_f = np.argmin(np.abs(run_st['sf'] - tk.S_MID)); i_r = np.argmin(np.abs(run_st['s'] - tk.S_MID))
print('Stanley 2 m/s, middle of the arc: front axle error', round(run_st['ef'][i_f], 4), '(when the front axle is there);',
      'rear axle error', round(run_st['e'][i_r], 4), '(when the rear axle is there)')

rear-axle circle when the front axle is on the 10 m arc: 9.682 m, so 0.318 m inside
steering on the arc: asin(2.5/10) = 14.48 deg; tan form atan(2.5/9.682) = 14.48 deg
Stanley 2 m/s, middle of the arc: front axle error 0.004 (when the front axle is there); rear axle error 0.3167 (when the rear axle is there)


## 5. Tuning the gain k at 5 m/s, with and without the 0.4 s steering lag (Note, Section 4.1)

In [7]:
for tau in (0.0, 0.4):
    for k in (0.5, 1.0, 2.5, 5.0):
        r = tk.simulate(tk.law('stanley', k=k), v=tk.V_FAST, tau=tau)
        m = report(f'k = {k}, lag {tau} s', r, 'front')

k = 0.5, lag 0.0 s                 [front] within 0.1 m after  32.28 m | past the line 0.000 m | arc: largest +0.526, middle +0.2609 | last straight +0.137 | steering up to 14.9 deg | 6.97 s


k = 1.0, lag 0.0 s                 [front] within 0.1 m after  15.39 m | past the line 0.000 m | arc: largest +0.276, middle +0.0649 | last straight +0.028 | steering up to 14.7 deg | 7.03 s


k = 2.5, lag 0.0 s                 [front] within 0.1 m after   7.79 m | past the line 0.000 m | arc: largest +0.040, middle -0.0016 | last straight +0.012 | steering up to 22.6 deg | 7.07 s
k = 5.0, lag 0.0 s                 [front] within 0.1 m after   5.12 m | past the line 0.000 m | arc: largest -0.006, middle -0.0010 | last straight +0.008 | steering up to 26.6 deg | 7.07 s


k = 0.5, lag 0.4 s                 [front] within 0.1 m after  13.06 m | past the line 0.000 m | arc: largest +0.479, middle -0.1661 | last straight +0.450 | steering up to 17.3 deg | 7.06 s
k = 1.0, lag 0.4 s                 [front] within 0.1 m after  10.66 m | past the line 0.000 m | arc: largest -0.262, middle -0.1775 | last straight +0.373 | steering up to 17.0 deg | 7.09 s


k = 2.5, lag 0.4 s                 [front] within 0.1 m after   6.22 m | past the line 0.237 m | arc: largest -0.131, middle -0.1007 | last straight +0.278 | steering up to 15.7 deg | 7.09 s
k = 5.0, lag 0.4 s                 [front] within 0.1 m after   5.40 m | past the line 0.458 m | arc: largest -0.324, middle -0.1234 | last straight +0.315 | steering up to 22.3 deg | 7.11 s


## 6. The softening speed (Note, Section 4.2)

In [8]:
k, ks = 1.0, 1.0
for v in (0.25, 0.5, 2.0, 5.0):
    print(f'v = {v}: gain k/v = {k/v:.3f} per m, softened k/(ks + v) = {k/(ks + v):.3f} per m')
noise = 0.05
for v in (0.5,):
    print('steering from a 0.05 m jump in the measured error at 0.5 m/s: without softening',
          round(deg(np.arctan(k * noise / v)), 2), 'deg; with', round(deg(np.arctan(k * noise / (ks + v))), 2), 'deg')

v = 0.25: gain k/v = 4.000 per m, softened k/(ks + v) = 0.800 per m
v = 0.5: gain k/v = 2.000 per m, softened k/(ks + v) = 0.667 per m
v = 2.0: gain k/v = 0.500 per m, softened k/(ks + v) = 0.333 per m
v = 5.0: gain k/v = 0.200 per m, softened k/(ks + v) = 0.167 per m
steering from a 0.05 m jump in the measured error at 0.5 m/s: without softening 5.71 deg; with 1.91 deg


## 7. Rear-wheel feedback (Note, Section 5)

In [9]:
o = tk.rear_wheel(*tk.START, tk.V_BASE)
print('start: turn rate', round(o['omega'], 3), 'rad/s (curvature', round(o['ff'], 3), ', heading', round(o['fb_h'], 3),
      ', lateral', round(o['fb_e'], 3), '); steering asked', round(deg(o['delta']), 2), 'deg')
xs, ys, hs, _ = tk.course(tk.S_MID)
o = tk.rear_wheel(float(xs), float(ys), float(hs), tk.V_BASE)
print('on the arc, on the line: turn rate', round(o['omega'], 3), 'rad/s; steering', round(deg(o['delta']), 2), 'deg')
for v in (2.0, 5.0):
    r = tk.simulate(tk.law('rear'), v=v)
    i = np.argmax(np.abs(r['e']) < 0.1)
    print(f'{v} m/s: within 0.1 m after {r["s"][i]:.2f} m, {r["t"][i]:.2f} s')
    report(f'rear-wheel {v} m/s', r)
wn = np.sqrt(tk.KE_RW); zeta = tk.KTH_RW / (2 * wn)
print('per metre: natural frequency', round(wn, 4), 'per m, damping ratio', round(zeta, 4),
      'overshoot', round(np.exp(-np.pi * zeta / np.sqrt(1 - zeta**2)), 4))

start: turn rate -1.0 rad/s (curvature 0.0 , heading -0.0 , lateral -1.0 ); steering asked -51.34 deg
on the arc, on the line: turn rate 0.2 rad/s; steering 14.04 deg


2.0 m/s: within 0.1 m after 4.09 m, 2.10 s
rear-wheel 2.0 m/s                 [rear] within 0.1 m after   4.09 m | past the line 0.039 m | arc: largest -0.006, middle +0.0002 | last straight +0.005 | steering up to 26.6 deg | 17.92 s
5.0 m/s: within 0.1 m after 4.03 m, 0.83 s
rear-wheel 5.0 m/s                 [rear] within 0.1 m after   4.03 m | past the line 0.039 m | arc: largest -0.007, middle +0.0003 | last straight +0.011 | steering up to 26.6 deg | 7.17 s
per metre: natural frequency 0.7071 per m, damping ratio 0.7071 overshoot 0.0432


## 8. Reversing (Note, Section 5.1): Stanley's front axle trails and the car turns round; rear-wheel feedback returns

In [10]:
start_rev = (8.0, 1.0, 0.0)
rev = {}
for name in ('stanley', 'rear'):
    r = tk.simulate(tk.law(name), v=-2.0, start=start_rev, s_stop=1e9, t_max=8.0)
    rev[name] = r
    print(name, ': lateral error every second', np.round(r['e'][::100], 2))
    print('   heading every second (deg)', np.round(deg(r['th'][::100]), 1))
r = rev['rear']; i = np.argmax(np.abs(r['e']) < 0.1)
print('rear-wheel feedback in reverse: within 0.1 m after', round(r['t'][i], 2), 's,', round(8 - r["x"][i], 2), 'm driven back')
print('largest past the line', round(-r['e'].min(), 3))

stanley : lateral error every second [ 1.    0.66 -0.4  -2.04 -3.98 -5.93 -7.58 -8.66 -9.  ]
   heading every second (deg) [  0.   21.   43.9  66.9  89.8 112.7 135.6 158.5 179.2]
rear : lateral error every second [ 1.    0.63  0.13 -0.03 -0.03 -0.01  0.    0.    0.  ]
   heading every second (deg) [ 0.  17.9  9.2  1.2 -0.7 -0.4 -0.1  0.   0. ]
rear-wheel feedback in reverse: within 0.1 m after 2.1 s, 4.09 m driven back
largest past the line 0.039
